# Qwen3.5-4B retrieval-augmented structured extraction

This notebook merges the W&B LoRA adapter into a verified local Qwen3.5 checkpoint, then evaluates it with retrieval-augmented inputs downloaded from W&B. The prepared retrieval corpus comes **only** from `notebooks/dataset/rag/train.parquet`; the 68 untouched rows from `notebooks/dataset/rag/test.parquet` expand to **2,108 section requests (68 × 31)**. Each request asks for exactly one canonical section while retaining the full untouched test document. No `sft/test` data and no gold-sliced test spans are used in prompts.

The retrieval design is a text-only adaptation of *RAG-Anything: All-in-One RAG Framework* (`RAG/2510.12323v1.pdf`). Each training judgment was decomposed into document and canonical-section nodes. Normalized sparse vector similarity was fused with structural document-to-section navigation, and the retrieved section examples were assembled into compact, provenance-labelled contexts before upload. The model must still copy every predicted value verbatim from the current test document; retrieved training values are examples of structure and location, never facts to copy.

The workflow has two phases: merge and verify, then restart only the Python kernel to release merge-phase GPU memory before evaluation. Runtime artifacts remain under `/content` across that kernel restart. The generated Parquet contains one row per test-document section, retrieval provenance, JSON validity, requested-section schema validity, and extractive-grounding metrics.


## 1. Install dependencies and restart once

Run this cell once. It installs the model runtime plus the dataset, embedding, and persistent vector-database dependencies. After Colab reconnects, continue at Section 2 and do not rerun this cell.


In [ ]:
%pip install -q -U uv
!uv pip install -q -U unsloth 'wandb>=0.21.0' 'weave>=0.52.0' 'safetensors>=0.5.0' 'pandas>=2.2.0' 'pyarrow>=17.0.0'
!uv pip install -q -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly
%pip uninstall -q -y torchaudio

print('Dependencies installed. Restarting the Python kernel so compiled packages load consistently.')
print('After reconnecting, continue at Section 2; do not rerun this install cell.')
from IPython import get_ipython
install_kernel = getattr(get_ipython(), 'kernel', None)
if install_kernel is None:
    raise RuntimeError('No active IPython kernel. Restart the Python kernel manually before importing Unsloth.')
install_kernel.do_shutdown(restart=True)


## 2. Configuration and local-Colab paths

The train/test split names are explicit invariants. Retrieval is deterministic and precomputed; the W&B artifact records exact prompt hashes, token budgets, the index fingerprint, and retrieved node IDs for every prediction.


In [ ]:
from __future__ import annotations

import gc
import json
import os
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import wandb
from safetensors import safe_open
from tqdm.auto import tqdm

BASE_MODEL = 'Qwen/Qwen3.5-4B'
MERGE_MAX_SEQUENCE_LENGTH = 49_152
MODEL_NATIVE_MAX_LENGTH = 262_144
MAX_GENERATION_TOKENS_CAP = 96_000
MIN_GENERATION_TOKENS = 1_024
CONTEXT_ALIGNMENT = 256
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = -1
MIN_P = 0.0
PRESENCE_PENALTY = 0.0
REPETITION_PENALTY = 1.0
ENABLE_THINKING = False
SEED = 3407
REQUESTED_BATCH_SIZE = 4
ENGINE_OVERHEAD_RESERVE_GIB = 8.0
GPU_MEMORY_UTILIZATION = 0.95
CACHE_PACKING_FRACTION = 0.90
TEST_LIMIT: int | None = None

DATASET_REPO = 'Haeryz/putusan-structured-extraction'
DATASET_CONFIG = 'rag'
REFERENCE_SPLIT = 'train'
EVALUATION_SPLIT = 'test'
RAG_RETRIEVAL_MODEL = 'sklearn-tfidf-word-1-2-v1'
RAG_INDEX_VERSION = 'rag-anything-text-v2-sparse-exact'
RAG_CHUNK_CHARS = 1_800
RAG_CHUNK_OVERLAP = 200
RAG_DOCUMENT_CANDIDATES = 12
RAG_SECTION_CANDIDATES = 12
RAG_REFERENCES_PER_SECTION = 1
RAG_REFERENCE_CHARS_PER_SECTION = 700
RAG_MAX_REFERENCE_CHARS = 24_000

WANDB_ENTITY = 'haeriz42069-universitas-muhammadiyah-malang'
WANDB_PROJECT = 'Sinergi-training'
ADAPTER_ARTIFACT = f'{WANDB_ENTITY}/{WANDB_PROJECT}/qwen3-5-4b-lora:v0'
RAG_EVAL_INPUT_ARTIFACT = f'{WANDB_ENTITY}/{WANDB_PROJECT}/qwen3-5-4b-rag-test-eval-inputs-no-thinking:v0'

IN_COLAB = Path('/content').is_dir() and 'COLAB_RELEASE_TAG' in os.environ
LOCAL_ROOT = Path('/content/qwen3-5-4b-rag-evaluation') if IN_COLAB else Path('artifacts/qwen3-5-4b-rag-evaluation')
ADAPTER_ROOT = LOCAL_ROOT / 'adapter-artifact'
MERGED_MODEL_DIR = LOCAL_ROOT / 'merged-bf16'
PRECOMPUTED_INPUT_ROOT = LOCAL_ROOT / 'precomputed-rag-eval-inputs'
OUTPUT_PARQUET = LOCAL_ROOT / 'qwen3-5-4b-rag-test-outputs.parquet'

for directory in (LOCAL_ROOT, ADAPTER_ROOT, MERGED_MODEL_DIR, PRECOMPUTED_INPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

assert REFERENCE_SPLIT == 'train' and EVALUATION_SPLIT == 'test'
assert DATASET_CONFIG == 'rag'
assert REQUESTED_BATCH_SIZE >= 1
assert TEST_LIMIT is None or TEST_LIMIT >= 1
assert RAG_CHUNK_CHARS > RAG_CHUNK_OVERLAP >= 0
assert RAG_REFERENCES_PER_SECTION >= 1
assert 1 <= MIN_GENERATION_TOKENS <= MAX_GENERATION_TOKENS_CAP < MODEL_NATIVE_MAX_LENGTH
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime.'
print(f'Local runtime root: {LOCAL_ROOT}')
print(f'Prepared RAG input cache: {PRECOMPUTED_INPUT_ROOT}')
print(f'Offline test-output backup: {OUTPUT_PARQUET}')


## 3. Authenticate for the merge phase

Colab Secrets may provide `WANDB_API_KEY` and `HF_TOKEN`; Google Drive is never imported or mounted. This phase only authenticates so the adapter can be downloaded. W&B evaluation and Weave tracing are initialized after the kernel restart.

In [ ]:
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def runtime_secret(name: str) -> str | None:
    if value := os.getenv(name):
        return value
    if userdata is not None:
        try:
            return userdata.get(name)
        except Exception:
            pass
    return None

wandb_key = runtime_secret('WANDB_API_KEY')
if not wandb_key:
    raise RuntimeError('Set WANDB_API_KEY in Colab Secrets or the environment.')
wandb.login(key=wandb_key, relogin=True)
hf_token = runtime_secret('HF_TOKEN')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

print('Authenticated. Continue to the merge cell.')

## 4. Download the adapter and merge it locally to verified BF16/FP32

Prerequisite: after the dependency restart, run the code cells in **Section 2** and **Section 3** before this cell. Do not jump directly from installation to the merge.

In [ ]:
required_merge_state = {
    'Path', 'wandb', 'ADAPTER_ARTIFACT', 'ADAPTER_ROOT', 'BASE_MODEL',
    'MERGE_MAX_SEQUENCE_LENGTH', 'MERGED_MODEL_DIR', 'torch', 'gc', 'json',
    'Counter', 'tqdm', 'safe_open', 'wandb_key',
}
missing_merge_state = sorted(name for name in required_merge_state if name not in globals())
if missing_merge_state:
    raise RuntimeError(
        'Merge prerequisites are missing after the kernel restart: '
        f'{missing_merge_state}. Run the Section 2 configuration cell and Section 3 '
        'authentication cell, then rerun this merge cell.'
    )

from unsloth import FastLanguageModel

artifact_root = Path(wandb.Api().artifact(ADAPTER_ARTIFACT).download(root=str(ADAPTER_ROOT)))
adapter_dir = artifact_root / 'adapter'
if not (adapter_dir / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'{ADAPTER_ARTIFACT} has no adapter/adapter_config.json')
adapter_config = json.loads((adapter_dir / 'adapter_config.json').read_text(encoding='utf-8'))
if (adapter_base := adapter_config.get('base_model_name_or_path')) and adapter_base != BASE_MODEL:
    raise RuntimeError(f'Adapter base {adapter_base!r} != {BASE_MODEL!r}')

if not (MERGED_MODEL_DIR / 'config.json').is_file():
    merge_model, merge_tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(adapter_dir), max_seq_length=MERGE_MAX_SEQUENCE_LENGTH,
        dtype=torch.bfloat16, load_in_4bit=True, text_only=True,
    )
    merge_model.save_pretrained_merged(
        str(MERGED_MODEL_DIR), merge_tokenizer, save_method='merged_16bit',
        maximum_memory_usage=0.65, safe_serialization=None,
    )
    del merge_model, merge_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('Reusing the merged checkpoint already present in this local runtime.')

dtype_counts: Counter[str] = Counter()
shards = sorted(MERGED_MODEL_DIR.glob('*.safetensors'))
if not shards:
    raise FileNotFoundError('Merged model has no safetensors shards.')
for shard in tqdm(shards, desc='Verifying merged tensor dtypes'):
    with safe_open(shard, framework='pt', device='cpu') as handle:
        for name in handle.keys():
            dtype_counts[str(handle.get_slice(name).get_dtype())] += 1
merged_config = json.loads((MERGED_MODEL_DIR / 'config.json').read_text(encoding='utf-8'))
merged_text_config = merged_config.get('text_config', merged_config)
declared_model_dtype = merged_text_config.get('dtype', merged_text_config.get('torch_dtype'))
if declared_model_dtype not in {None, 'bfloat16'}:
    raise RuntimeError(f'Expected Qwen3.5 dtype=bfloat16 or omitted, found {declared_model_dtype!r}')
declared_ssm_dtype = merged_text_config.get('mamba_ssm_dtype')
if declared_ssm_dtype not in {None, 'float32'}:
    raise RuntimeError(
        f'Expected Qwen3.5 mamba_ssm_dtype=float32 or omitted, found {declared_ssm_dtype!r}'
    )
floating = {name for name in dtype_counts if name in {'F16', 'BF16', 'F32', 'F64'}}
if 'BF16' not in floating or not floating <= {'BF16', 'F32'}:
    raise RuntimeError(
        f'Expected BF16 weights with optional FP32 recurrent tensors, found {sorted(floating)}'
    )
print(f'BF16/FP32 verification passed: {dict(dtype_counts)}')

## 5. Restart the kernel after the merge

The verified merged checkpoint is already stored under `/content/qwen3-5-4b-rag-evaluation/merged-bf16`, which survives a Colab **kernel/session restart**. Run the next cell once. It restarts only the Python kernel so every Unsloth, Transformers, PyTorch, CUDA-context, and merged-model allocation is released from the A100. Do **not** disconnect, delete, or factory-reset the Colab runtime.

When Colab reconnects, continue directly at **Phase 2 - fresh-kernel RAG bootstrap**. Do not rerun the merge cell.

In [ ]:
if not (MERGED_MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError('Merged checkpoint config is missing; do not restart yet.')
if not any(MERGED_MODEL_DIR.glob('*.safetensors')):
    raise FileNotFoundError('Merged checkpoint shards are missing; do not restart yet.')
print(f'Checkpoint safely stored at {MERGED_MODEL_DIR.resolve()}')
print('Restarting only the Python kernel to release all merge-phase VRAM...')
from IPython import get_ipython
kernel = getattr(get_ipython(), 'kernel', None)
if kernel is None:
    raise RuntimeError('No active IPython kernel. Use Runtime > Restart session manually.')
kernel.do_shutdown(restart=True)

# Phase 2 - fresh-kernel RAG bootstrap

Start here after the kernel reconnects. This cell restores configuration and authentication without importing Unsloth or loading merge-phase weights. The next section downloads fully retrieved and tokenized `rag/test` prompts from W&B, validates their `rag/train` provenance, and then starts vLLM directly.


In [ ]:
import gc
import hashlib
import json
import math
import os
import re
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import wandb
from tqdm.auto import tqdm

BASE_MODEL = 'Qwen/Qwen3.5-4B'
MODEL_NATIVE_MAX_LENGTH = 262_144
MAX_GENERATION_TOKENS_CAP = 96_000
MIN_GENERATION_TOKENS = 1_024
CONTEXT_ALIGNMENT = 256
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = -1
MIN_P = 0.0
PRESENCE_PENALTY = 0.0
REPETITION_PENALTY = 1.0
ENABLE_THINKING = False
SEED = 3407
REQUESTED_BATCH_SIZE = 4
MAX_NUM_BATCHED_TOKENS = 32_768
SUBMISSION_CHUNK_SIZE = 31
THROUGHPUT_SAFETY_FACTOR = 0.85
ENGINE_OVERHEAD_RESERVE_GIB = 8.0
GPU_MEMORY_UTILIZATION = 0.95
CACHE_PACKING_FRACTION = 0.90
TEST_LIMIT: int | None = None

DATASET_REPO = 'Haeryz/putusan-structured-extraction'
DATASET_CONFIG = 'rag'
REFERENCE_SPLIT = 'train'
EVALUATION_SPLIT = 'test'
RAG_RETRIEVAL_MODEL = 'sklearn-tfidf-word-1-2-v1'
RAG_INDEX_VERSION = 'rag-anything-text-v2-sparse-exact'
RAG_CHUNK_CHARS = 1_800
RAG_CHUNK_OVERLAP = 200
RAG_DOCUMENT_CANDIDATES = 12
RAG_SECTION_CANDIDATES = 12
RAG_REFERENCES_PER_SECTION = 1
RAG_REFERENCE_CHARS_PER_SECTION = 700
RAG_MAX_REFERENCE_CHARS = 24_000

CANONICAL_SECTIONS = [
    'judul', 'nomor_putusan', 'irah_irah', 'nama_pengadilan_negeri',
    'keterangan_perkara', 'nama_lengkap', 'tempat_lahir', 'umur_tanggal_lahir',
    'jenis_kelamin', 'kebangsaan', 'tempat_tinggal', 'agama', 'pekerjaan',
    'penangkapan', 'penahanan', 'tuntutan', 'dakwaan', 'saksi', 'ahli',
    'terdakwa', 'surat', 'petunjuk_barang_bukti', 'fakta_hukum',
    'pertimbangan_hukum', 'amar_putusan', 'hari', 'tanggal', 'tahun',
    'siapa_yang_memutus', 'panitera_pengganti', 'tanda_tangan_majelis',
]

# Authoritative prompt recovered from the dataset/W&B prompt artifact.  Keep it
# explicit here: the tokenizer only supplies chat-template control tokens.
ORIGINAL_DATASET_SYSTEM_PROMPT = '''Anda adalah pengekstrak terstruktur putusan pengadilan Indonesia. Diberikan badan teks putusan, keluarkan SATU objek JSON dengan tepat 31 kunci bagian (dalam urutan kanonik). Setiap nilai adalah daftar kutipan verbatim (extractive) yang disalin persis dari teks sumber — jangan pernah memparafrasekan, meringkas, atau mengarang. Jika sebuah bagian tidak ada, gunakan daftar kosong dan cantumkan kuncinya di 'empty_sections'. Kunci bagian, dalam urutan: judul, nomor_putusan, irah_irah, nama_pengadilan_negeri, keterangan_perkara, nama_lengkap, tempat_lahir, umur_tanggal_lahir, jenis_kelamin, kebangsaan, tempat_tinggal, agama, pekerjaan, penangkapan, penahanan, tuntutan, dakwaan, saksi, ahli, terdakwa, surat, petunjuk_barang_bukti, fakta_hukum, pertimbangan_hukum, amar_putusan, hari, tanggal, tahun, siapa_yang_memutus, panitera_pengganti, tanda_tangan_majelis.'''

# Exact section-boundary guidance used by the original section evaluation.
SECTION_GUIDANCE = {
    'judul': 'Judul: ambil judul PUTUSAN/P U T U S A N di awal dokumen; berhenti sebelum nomor perkara.',
    'nomor_putusan': 'Nomor Putusan: ambil nomor perkara yang mengikuti judul; jangan sertakan nama pengadilan atau irah-irah.',
    'irah_irah': 'Irah-irah: ambil formula DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA, termasuk variasi spasi/OCR, tanpa bagian pengadilan sesudahnya.',
    'nama_pengadilan_negeri': 'Nama Pengadilan Negeri: ambil nama pengadilan tingkat pertama pada kalimat pembuka, bukan seluruh uraian jenis perkara.',
    'keterangan_perkara': 'Keterangan Perkara: ambil uraian bahwa pengadilan mengadili perkara, acara pemeriksaan, tingkat, dan subjek perkara sampai sebelum identitas.',
    'nama_lengkap': 'Nama Lengkap: ambil hanya nilai identitas pada label 1. Nama lengkap. Untuk beberapa Anak/Terdakwa, pertahankan semua nilai dan label I/II/III dalam urutan dokumen.',
    'tempat_lahir': 'Tempat Lahir: ambil hanya nilai pada label 2. Tempat lahir untuk setiap subjek; berhenti sebelum umur/tanggal lahir.',
    'umur_tanggal_lahir': 'Umur/Tanggal Lahir: ambil hanya nilai umur dan/atau tanggal lahir pada label 3 untuk setiap subjek.',
    'jenis_kelamin': 'Jenis Kelamin: ambil hanya nilai pada label 4 untuk setiap subjek.',
    'kebangsaan': 'Kebangsaan: ambil hanya nilai pada label 5; Kewarganegaraan/nasionalitas adalah alias. Jangan menyerap Pendidikan yang kadang muncul sesudahnya.',
    'tempat_tinggal': 'Tempat Tinggal: ambil hanya alamat pada label 6 untuk setiap subjek; berhenti sebelum Agama.',
    'agama': 'Agama: ambil hanya nilai pada label 7 untuk setiap subjek; berhenti sebelum Pekerjaan.',
    'pekerjaan': 'Pekerjaan: ambil hanya nilai pada label 8 untuk setiap subjek; berhenti sebelum kalimat penangkapan/penahanan atau prosedur berikutnya.',
    'penangkapan': 'Penangkapan: ambil hanya kalimat/perintah penangkapan beserta tanggal dan referensinya. Jangan memasukkan satu pun tahap penahanan.',
    'penahanan': 'Penahanan: ambil seluruh tahap dan perpanjangan penahanan, termasuk penangguhan, pembantaran, pengalihan, atau penahanan dalam perkara lain. Jangan memasukkan penangkapan.',
    'tuntutan': 'Tuntutan: mulai pada pembacaan tuntutan pidana dan salin seluruh amar tuntutan bernomor, pidana/denda yang diminta, status tahanan, barang bukti/restitusi bila ada, dan biaya; berhenti pada pembelaan atau dakwaan berikutnya.',
    'dakwaan': 'Dakwaan: mulai saat Terdakwa/Anak didakwa berdasarkan surat dakwaan dan ambil lengkap semua bentuk tunggal, alternatif, subsidair, kumulatif, atau gabungan beserta uraian perbuatan dan pasal; pada acara singkat, catatan dakwaan adalah dakwaan.',
    'saksi': 'Saksi: ambil seluruh keterangan saksi penuntut, korban/anak korban/anak saksi, saksi meringankan (a de charge), dan verbalisan, termasuk nama, sumpah, butir keterangan, serta tanggapan Terdakwa/Anak.',
    'ahli': 'Ahli: ambil seluruh keterangan ahli penuntut maupun pembela, termasuk keterangan ahli yang dibacakan di persidangan; jangan mengisi dari keterangan saksi biasa.',
    'terdakwa': 'Terdakwa: ambil keterangan Terdakwa/Para Terdakwa atau Anak sendiri di persidangan, dari formula telah memberikan keterangan sampai sebelum kelompok alat bukti/fakta berikutnya.',
    'surat': 'Surat: ambil alat bukti surat, dokumen, dan alat bukti elektronik pada bagian Surat/bukti surat; jangan mencampurnya dengan daftar barang bukti fisik.',
    'petunjuk_barang_bukti': 'Petunjuk/Barang Bukti: ambil inventaris barang bukti yang diajukan Penuntut Umum. Pembahasan hukum tentang nasib/disposisi barang bukti yang muncul kemudian bukan inventaris ini.',
    'fakta_hukum': 'Fakta Hukum: mulai pada formula berdasarkan alat bukti diperoleh fakta hukum/fakta-fakta hukum dan ambil daftar faktanya; berhenti tepat sebelum Majelis mulai analisis hukum atau unsur.',
    'pertimbangan_hukum': 'Pertimbangan Hukum: mulai ketika Majelis mempertimbangkan dakwaan/unsur. Sertakan analisis semua unsur, kesimpulan pembuktian, alasan pembenar/pemaaf, sanksi, tahanan, restitusi/kompensasi, disposisi barang bukti, keadaan memberatkan/meringankan, biaya, dan Mengingat/Memperhatikan; berhenti sebelum MENGADILI.',
    'amar_putusan': 'Amar Putusan: mulai pada MENGADILI/M E N G A D I L I dan ambil setiap perintah bernomor—status terbukti/bebas/lepas, pidana, denda/restitusi, tahanan, barang bukti, dan biaya—sampai sebelum Demikianlah diputuskan.',
    'hari': 'Hari: ambil nama hari tanggal musyawarah putusan dari formula Demikianlah diputuskan, bukan hari pengucapan bila berbeda.',
    'tanggal': 'Tanggal: ambil tanggal musyawarah putusan persis seperti span sumber yang sudah dipotong; jangan mengubah atau membuang tahunnya bila tercantum.',
    'tahun': 'Tahun: ambil hanya tahun musyawarah putusan dari formula penutup.',
    'siapa_yang_memutus': 'Siapa yang Memutus: salin seluruh span keputusan tentang hakim yang memutus, termasuk formula dan peran yang sudah tercakup dalam span; jangan mempersempit span menjadi nama saja.',
    'panitera_pengganti': 'Panitera Pengganti: ambil nama Panitera/Panitera Pengganti yang membantu persidangan dari paragraf penutup atau blok tanda tangan.',
    'tanda_tangan_majelis': 'Tanda Tangan Majelis: ambil blok tanda tangan/nama Hakim Ketua, Hakim Anggota, dan Panitera Pengganti pada akhir dokumen, mempertahankan susunan aslinya.',
}
CORPUS_GUIDANCE = {
    'Anak': 'Korpus Anak: subjek adalah Anak, bukan Terdakwa dewasa. Identitas jamak tetap berurutan. Penahanan mencakup seluruh LPAS/LPKS dan perpanjangannya. Saksi mencakup anak korban dan anak saksi. Keterangan Anak sendiri masuk terdakwa. Laporan Penelitian Kemasyarakatan, rekomendasi Pembimbing Kemasyarakatan, orang tua/wali/pendamping, kepentingan terbaik Anak, pilihan tindakan/pidana, dan alasan sanksi Anak masuk pertimbangan_hukum bila berada dalam analisis Majelis. Dakwaan acara singkat dapat disebut catatan dakwaan.',
    'Asusila': 'Korpus Asusila/Pidana Biasa: subjek adalah Terdakwa/Para Terdakwa dan semua identitas harus berurutan. Penahanan Rutan dapat memuat Penyidik, perpanjangan Penuntut Umum, Ketua PN, Penuntut Umum, Hakim/Majelis, Ketua PT, serta tahap lanjutan. Dakwaan dapat tunggal, alternatif, subsidairitas, kumulatif, atau gabungan. Saksi mencakup korban, a de charge, dan verbalisan. Surat mencakup dokumen/elektronik. Restitusi atau kompensasi masuk pertimbangan atau amar sesuai letaknya.',
    'TPPO': 'Korpus TPPO: pertahankan semua Terdakwa dalam urutan. Pada penahanan cari Khusus Penahanan Tindak Pidana TPPO dan ambil seluruh tahap sampai perpanjangan kedua Ketua PT bila ada. Tuntutan dapat memuat restitusi. Pertimbangan TPPO mencakup bagian KHUSUS PERKARA TPPO, restitusi, tenggang pembayaran, penyitaan/lelang, pidana pengganti, serta disposisi barang bukti. Amar harus memuat semua perintah restitusi termasuk tenggang 14 hari, penyitaan/lelang harta, atau pidana pengganti. Jangan pindahkan pembahasan disposisi barang bukti ke inventaris petunjuk_barang_bukti.',
}

def system_prompt_for(corpus: str) -> str:
    if corpus not in CORPUS_GUIDANCE:
        raise ValueError(f'Unknown corpus for prompt guidance: {corpus!r}')
    section_contract = '\n'.join(
        f'- {section}: {SECTION_GUIDANCE[section]}' for section in CANONICAL_SECTIONS
    )
    return (
        'Anda adalah pengekstrak terstruktur putusan pengadilan Indonesia. '
        'Setiap permintaan mengekstrak SATU bagian kanonik dari TARGET DOCUMENT lengkap. '
        'Keluarkan SATU objek JSON saja tanpa markdown, penjelasan, analisis, reasoning, atau teks lain. '
        'Bentuk wajib: {\"sections\": {\"nama_bagian\": [\"kutipan\"]}, \"empty_sections\": []}. '
        'Objek sections wajib berisi tepat satu kunci yang diminta. Setiap string harus kutipan verbatim '
        'dan kontigu dari TARGET DOCUMENT: jangan meringkas, memparafrasekan, memperbaiki OCR, '
        'menormalkan ejaan/spasi, menggabungkan potongan tak-kontigu, mengarang, atau menyalin fakta '
        'dari TRAIN REFERENCE. Jika bagian tidak ada, nilainya [] dan empty_sections berisi nama bagian; '
        'jika ada kutipan, empty_sections harus [].'
        + '\n\nBatas dan arti setiap bagian:\n' + section_contract
        + f'\n\nPanduan korpus {corpus}: {CORPUS_GUIDANCE[corpus]}'
    )

WANDB_ENTITY = 'haeriz42069-universitas-muhammadiyah-malang'
WANDB_PROJECT = 'Sinergi-training'
ADAPTER_ARTIFACT = f'{WANDB_ENTITY}/{WANDB_PROJECT}/qwen3-5-4b-lora:v0'
RAG_EVAL_INPUT_ARTIFACT = f'{WANDB_ENTITY}/{WANDB_PROJECT}/qwen3-5-4b-rag-test-eval-inputs-no-thinking:v0'
IN_COLAB = Path('/content').is_dir() and 'COLAB_RELEASE_TAG' in os.environ
LOCAL_ROOT = Path('/content/qwen3-5-4b-rag-evaluation') if IN_COLAB else Path('artifacts/qwen3-5-4b-rag-evaluation')
ADAPTER_ROOT = LOCAL_ROOT / 'adapter-artifact'
MERGED_MODEL_DIR = LOCAL_ROOT / 'merged-bf16'
PRECOMPUTED_INPUT_ROOT = LOCAL_ROOT / 'precomputed-rag-eval-inputs'
OUTPUT_PARQUET = LOCAL_ROOT / 'qwen3-5-4b-rag-test-outputs.parquet'

assert DATASET_CONFIG == 'rag'
assert REFERENCE_SPLIT == 'train' and EVALUATION_SPLIT == 'test'
assert len(CANONICAL_SECTIONS) == 31 and len(set(CANONICAL_SECTIONS)) == 31
assert MAX_NUM_BATCHED_TOKENS >= REQUESTED_BATCH_SIZE
assert SUBMISSION_CHUNK_SIZE >= REQUESTED_BATCH_SIZE
assert TEST_LIMIT is None or TEST_LIMIT >= 1

if not (MERGED_MODEL_DIR / 'config.json').is_file() or not any(MERGED_MODEL_DIR.glob('*.safetensors')):
    raise FileNotFoundError(f'Merged checkpoint is incomplete at {MERGED_MODEL_DIR}; run Phase 1 first.')

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def runtime_secret(name: str) -> str | None:
    if value := os.getenv(name):
        return value
    if userdata is not None:
        try:
            return userdata.get(name)
        except Exception:
            pass
    return None

wandb_key = runtime_secret('WANDB_API_KEY')
if not wandb_key:
    raise RuntimeError('Set WANDB_API_KEY in Colab Secrets or the environment.')
wandb.login(key=wandb_key, relogin=True)
hf_token = runtime_secret('HF_TOKEN')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name='qwen3-5-4b-rag-test-evaluation',
    job_type='rag-evaluation',
    config={
        'base_model': BASE_MODEL,
        'adapter_artifact': ADAPTER_ARTIFACT,
        'dataset': f'{DATASET_REPO}/{DATASET_CONFIG}',
        'reference_split': REFERENCE_SPLIT,
        'evaluation_split': EVALUATION_SPLIT,
        'rag_input_artifact': RAG_EVAL_INPUT_ARTIFACT,
        'retrieval_model': RAG_RETRIEVAL_MODEL,
        'rag_index_version': RAG_INDEX_VERSION,
        'retrieval': 'dense section similarity + structural document-to-section expansion',
        'test_prompt_source': 'full input_text only; no gold test spans or answers',
        'enable_thinking': ENABLE_THINKING,
        'temperature': TEMPERATURE,
        'max_live_sequences': REQUESTED_BATCH_SIZE,
    },
)

if not torch.cuda.is_available():
    raise RuntimeError('Select an A100 GPU runtime before indexing and evaluation.')
free_bytes, total_bytes = torch.cuda.mem_get_info()
if free_bytes / total_bytes < 0.90:
    raise RuntimeError('Less than 90% VRAM is free. Restart the kernel again before Phase 2.')
print(f'Fresh GPU VRAM: {free_bytes / 2**30:.2f}/{total_bytes / 2**30:.2f} GiB free')
print(f'W&B run: {run.url}')


## 6. Download and expand the verified RAG retrieval bundles

This stage downloads the immutable W&B artifact `qwen3-5-4b-rag-test-eval-inputs-no-thinking:v0`. Its 68 document bundles were prepared from real `rag/test` documents after train-only retrieval over all 529 `rag/train` judgments; each bundle already contains exactly one retrieved reference for every canonical section. The notebook verifies the artifact and its recovered system prompt, then expands each bundle into 31 one-section requests with the explicit section and corpus contract above. It uses the tokenizer already saved inside the merged checkpoint to render and count prompts; nothing is uploaded and retrieval is not rerun.

The downloaded artifact is validated before use: file hashes, prompt hashes, split isolation, train-only provenance, row counts, retrieval settings, and thinking mode must all match this notebook.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

prepared_artifact = run.use_artifact(RAG_EVAL_INPUT_ARTIFACT)
prepared_dir = Path(prepared_artifact.download(root=str(PRECOMPUTED_INPUT_ROOT)))
PRECOMPUTED_PARQUET = prepared_dir / 'qwen3-5-4b-rag-test-eval-no-thinking.parquet'
TRAIN_MANIFEST_PARQUET = prepared_dir / 'qwen3-5-4b-rag-train-manifest.parquet'
PRECOMPUTED_SUMMARY = prepared_dir / 'qwen3-5-4b-rag-test-eval-no-thinking-summary.json'
for required_file in (PRECOMPUTED_PARQUET, TRAIN_MANIFEST_PARQUET, PRECOMPUTED_SUMMARY):
    if not required_file.is_file():
        raise FileNotFoundError(f'Incomplete prepared RAG artifact: {required_file}')

prepared_summary = json.loads(PRECOMPUTED_SUMMARY.read_text(encoding='utf-8'))
precomputed_eval = pd.read_parquet(PRECOMPUTED_PARQUET)
train_df = pd.read_parquet(TRAIN_MANIFEST_PARQUET)
required_columns = {
    'no', 'dataset_id', 'corpus', 'source_file', 'source_sha256', 'purpose', 'split',
    'input_text', 'target_json', 'prompt', 'prompt_sha256', 'prompt_tokens_estimate',
    'source_tokens', 'gold_tokens', 'max_new_tokens', 'sequence_token_budget',
    'retrieved_reference_count', 'retrieved_train_ids_json',
    'retrieved_node_ids_json', 'retrieval_scores_json',
    'retrieval_references_json', 'rag_index_fingerprint',
}
if missing := required_columns - set(precomputed_eval.columns):
    raise RuntimeError(f'Prepared RAG artifact is missing columns: {sorted(missing)}')
expected_summary = {
    'schema_version': 1, 'reference_split': REFERENCE_SPLIT,
    'evaluation_split': EVALUATION_SPLIT, 'reference_rows': 529,
    'evaluation_rows': 68, 'enable_thinking': ENABLE_THINKING,
    'embedding_model': RAG_RETRIEVAL_MODEL, 'rag_index_version': RAG_INDEX_VERSION,
}
for key, expected in expected_summary.items():
    if prepared_summary.get(key) != expected:
        raise RuntimeError(f'Prepared RAG summary {key}={prepared_summary.get(key)!r}, expected {expected!r}')
if sha256_file(PRECOMPUTED_PARQUET) != prepared_summary['prepared_parquet_sha256']:
    raise RuntimeError('Prepared RAG Parquet SHA-256 mismatch')
if sha256_file(TRAIN_MANIFEST_PARQUET) != prepared_summary['train_manifest_sha256']:
    raise RuntimeError('Prepared train manifest SHA-256 mismatch')
if len(precomputed_eval) != prepared_summary['evaluation_rows'] or precomputed_eval['dataset_id'].duplicated().any():
    raise RuntimeError('Prepared rag/test rows are incomplete or duplicated')
if len(train_df) != prepared_summary['reference_rows'] or train_df['id'].duplicated().any():
    raise RuntimeError('Prepared rag/train manifest is incomplete or duplicated')
if set(precomputed_eval['split']) != {EVALUATION_SPLIT} or set(precomputed_eval['purpose']) != {'rag'}:
    raise RuntimeError('Prepared evaluation rows are not exclusively rag/test')
if set(train_df['split']) != {REFERENCE_SPLIT} or set(train_df['purpose']) != {'rag'}:
    raise RuntimeError('Prepared reference manifest is not exclusively rag/train')
if set(precomputed_eval['dataset_id']) & set(train_df['id']):
    raise RuntimeError('Prepared RAG artifact has train/test ID leakage')
if set(precomputed_eval['source_sha256']) & set(train_df['source_sha256']):
    raise RuntimeError('Prepared RAG artifact has train/test source leakage')
calculated_prompt_hashes = precomputed_eval['prompt'].map(lambda value: hashlib.sha256(value.encode('utf-8')).hexdigest())
if not calculated_prompt_hashes.equals(precomputed_eval['prompt_sha256']):
    raise RuntimeError('Prepared RAG prompt SHA-256 validation failed')

def artifact_system_message(rendered_prompt: str) -> str:
    system_open, system_close = '<|im_start|>system\n', '<|im_end|>\n'
    if not rendered_prompt.startswith(system_open):
        raise RuntimeError('Prepared prompt does not start with a Qwen system turn')
    end = rendered_prompt.find(system_close, len(system_open))
    if end < 0:
        raise RuntimeError('Prepared prompt has no closing system-turn marker')
    return rendered_prompt[len(system_open):end]

artifact_system_prompts = set(precomputed_eval['prompt'].map(artifact_system_message))
if artifact_system_prompts != {ORIGINAL_DATASET_SYSTEM_PROMPT}:
    raise RuntimeError('W&B prompt system message differs from the explicit dataset contract')
reference_train_ids = set(train_df['id'])
retrieved_id_sets = precomputed_eval['retrieved_train_ids_json'].map(lambda value: set(json.loads(value)))
if invalid_ids := set().union(*retrieved_id_sets) - reference_train_ids:
    raise RuntimeError(f'Prepared retrieval provenance contains non-train IDs: {sorted(invalid_ids)[:5]}')
if (precomputed_eval['retrieved_reference_count'] <= 0).any():
    raise RuntimeError('At least one prepared test prompt has no retrieved train references')
if precomputed_eval['rag_index_fingerprint'].nunique() != 1:
    raise RuntimeError('Prepared rows contain multiple RAG index fingerprints')
index_fingerprint = prepared_summary['rag_index_fingerprint']
if precomputed_eval['rag_index_fingerprint'].iloc[0] != index_fingerprint:
    raise RuntimeError('Prepared row fingerprint differs from summary')
rag_index_node_count = int(prepared_summary['rag_index_node_count'])
prepared_input_artifact_name = prepared_artifact.qualified_name

if TEST_LIMIT is not None:
    precomputed_eval = precomputed_eval.iloc[:TEST_LIMIT].copy()
test_df = precomputed_eval[['dataset_id', 'source_sha256']].rename(columns={'dataset_id': 'id'}).copy()

# Expand 68 document-level retrieval bundles into 68 x 31 section requests.
# Phase 1 already saved this tokenizer; it is loaded locally for chat rendering
# and exact budgets only, and is never uploaded again.
from transformers import AutoTokenizer
prompt_tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR, trust_remote_code=False)

work: list[dict[str, Any]] = []
for artifact_row in precomputed_eval.to_dict('records'):
    row = {
        'id': artifact_row['dataset_id'], 'corpus': artifact_row['corpus'],
        'source_file': artifact_row['source_file'], 'source_sha256': artifact_row['source_sha256'],
        'purpose': artifact_row['purpose'], 'split': artifact_row['split'],
        'input_text': artifact_row['input_text'], 'target_json': artifact_row['target_json'],
    }
    target = json.loads(row['target_json'])
    sections = target.get('sections', {})
    if list(sections) != CANONICAL_SECTIONS:
        raise RuntimeError(f"Canonical target mismatch for {row['id']}")
    references = json.loads(artifact_row['retrieval_references_json'])
    references_by_section = {reference['section']: reference for reference in references}
    if list(references_by_section) != CANONICAL_SECTIONS:
        raise RuntimeError(f"Retrieval bundle is not one reference per section for {row['id']}")
    system_message = system_prompt_for(row['corpus'])
    for section in CANONICAL_SECTIONS:
        reference = references_by_section[section]
        # Target first preserves a long shared prefix across the 31 requests for
        # one document; the section-specific train reference follows it.
        question = f'''<target_document>
{row['input_text']}
</target_document>

<retrieved_training_reference section="{section}" corpus="{reference['corpus']}" source="{reference['source_file']}">
{reference['text']}
</retrieved_training_reference>

Bagian yang diminta: {section}. Referensi train hanya membantu mengenali batas bagian; jangan salin faktanya. Keluarkan tepat satu objek JSON untuk bagian {section}.'''
        prompt = prompt_tokenizer.apply_chat_template(
            [{'role': 'system', 'content': system_message}, {'role': 'user', 'content': question}],
            tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING,
        )
        spans = list(sections[section])
        gold_answer = json.dumps(
            {'sections': {section: spans}, 'empty_sections': [section] if not spans else []},
            ensure_ascii=False,
        )
        prompt_tokens = len(prompt_tokenizer.encode(prompt, add_special_tokens=False))
        gold_tokens = len(prompt_tokenizer.encode(gold_answer, add_special_tokens=False))
        max_new_tokens = min(MAX_GENERATION_TOKENS_CAP, max(64, math.ceil(gold_tokens * 1.05 + 32)))
        if max_new_tokens < gold_tokens:
            raise RuntimeError(f"Generation cap is below gold tokens for {row['id']}::{section}")
        if prompt_tokens + max_new_tokens > MODEL_NATIVE_MAX_LENGTH:
            raise RuntimeError(f"{row['id']}::{section} exceeds the native context")
        work.append({
            'no': len(work) + 1, 'document_no': int(artifact_row['no']),
            'dataset_id': f"{row['id']}::{section}", 'row': row, 'section': section,
            'question': question, 'prompt': prompt, 'gold_answer': gold_answer,
            'prompt_tokens_estimate': prompt_tokens,
            'source_tokens': int(artifact_row['source_tokens']), 'gold_tokens': gold_tokens,
            'max_new_tokens': max_new_tokens,
            'sequence_token_budget': prompt_tokens + max_new_tokens,
            'references': [reference],
        })
del prompt_tokenizer
gc.collect()
prompt_counts = pd.Series([item['prompt_tokens_estimate'] for item in work])
sequence_budgets = pd.Series([item['sequence_token_budget'] for item in work])
print(
    f'Built {len(work):,} section requests ({len(precomputed_eval)} documents x '
    f'{len(CANONICAL_SECTIONS)} sections) from {prepared_input_artifact_name}; '
    f'prompt tokens p50={prompt_counts.median():,.0f}, p95={prompt_counts.quantile(.95):,.0f}, '
    f'max={prompt_counts.max():,}; max sequence budget={sequence_budgets.max():,}. '
    'Prompts use the explicit section/corpus contract and the existing merged tokenizer.'
)


## 7. Size the Qwen3.5 hybrid cache from the retrieved prompts and start vLLM

The engine context length is derived from the section prompt and generation budgets. The small CPU tokenizer used to render/count prompts has already been deleted before vLLM starts; the retrieval model is never loaded. Each submission contains the 31 section requests for one document, with at most four live sequences. Documents run shortest first, and putting the target document before the section-specific reference lets vLLM reuse its long cached prefix.


In [ ]:
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
import io
import sys
from contextlib import contextmanager
from transformers import AutoConfig
from vllm import LLM, SamplingParams
import vllm.distributed.parallel_state as vllm_parallel_state

try:
    sys.stdout.fileno()
except (AttributeError, OSError, io.UnsupportedOperation):
    @contextmanager
    def notebook_safe_suppress_stdout():
        yield
    vllm_parallel_state.suppress_stdout = notebook_safe_suppress_stdout

official_qwen_config = AutoConfig.from_pretrained(BASE_MODEL, token=hf_token)
if official_qwen_config.model_type != 'qwen3_5':
    raise RuntimeError(f'Unexpected base config type: {official_qwen_config.model_type!r}')
official_qwen_config.save_pretrained(MERGED_MODEL_DIR)

MAX_MODEL_LENGTH = min(
    MODEL_NATIVE_MAX_LENGTH,
    ((max(item['sequence_token_budget'] for item in work) + CONTEXT_ALIGNMENT - 1) // CONTEXT_ALIGNMENT) * CONTEXT_ALIGNMENT,
)
if MAX_MODEL_LENGTH < max(item['sequence_token_budget'] for item in work):
    raise RuntimeError('A RAG sequence budget exceeds the native model context')

TENSOR_PARALLEL_SIZE = int(os.getenv('SINERGI_TENSOR_PARALLEL_SIZE', str(torch.cuda.device_count())))
if not 1 <= TENSOR_PARALLEL_SIZE <= torch.cuda.device_count():
    raise ValueError(f'Invalid tensor parallel size: {TENSOR_PARALLEL_SIZE}')
gpu_properties = [torch.cuda.get_device_properties(index) for index in range(TENSOR_PARALLEL_SIZE)]

saved_config = json.loads((MERGED_MODEL_DIR / 'config.json').read_text(encoding='utf-8'))
text_config = saved_config.get('text_config', saved_config)
layer_types = list(text_config.get('layer_types', []))
num_layers = int(text_config['num_hidden_layers'])
if layer_types:
    full_attention_layers = layer_types.count('full_attention')
    linear_attention_layers = layer_types.count('linear_attention')
else:
    full_attention_interval = int(text_config['full_attention_interval'])
    full_attention_layers = num_layers // full_attention_interval
    linear_attention_layers = num_layers - full_attention_layers
kv_bytes_per_token = (
    full_attention_layers * 2 * int(text_config['num_key_value_heads'])
    * int(text_config['head_dim']) * 2
)
ssm_element_bytes = 4 if text_config.get('mamba_ssm_dtype') == 'float32' else 2
delta_state_bytes_per_sequence = (
    linear_attention_layers * int(text_config['linear_num_value_heads'])
    * int(text_config['linear_key_head_dim'])
    * int(text_config['linear_value_head_dim']) * ssm_element_bytes
)
model_weight_bytes = sum(path.stat().st_size for path in MERGED_MODEL_DIR.glob('*.safetensors'))
per_gpu_weight_bytes = model_weight_bytes / TENSOR_PARALLEL_SIZE
smallest_gpu_bytes = min(props.total_memory for props in gpu_properties)
cache_budget_per_gpu = (
    smallest_gpu_bytes * GPU_MEMORY_UTILIZATION
    - per_gpu_weight_bytes - ENGINE_OVERHEAD_RESERVE_GIB * 2**30
)
if cache_budget_per_gpu <= 0:
    raise RuntimeError('No KV/state-cache budget remains after weights and engine reserve')
packing_cache_limit_per_gpu = cache_budget_per_gpu * CACHE_PACKING_FRACTION

def sequence_cache_bytes(item: dict[str, Any]) -> float:
    return (
        item['sequence_token_budget'] * kv_bytes_per_token + delta_state_bytes_per_sequence
    ) / TENSOR_PARALLEL_SIZE

for item in work:
    item['estimated_cache_bytes_per_gpu'] = sequence_cache_bytes(item)
    if item['estimated_cache_bytes_per_gpu'] > packing_cache_limit_per_gpu:
        raise RuntimeError(
            f"{item['row']['id']} requires {item['estimated_cache_bytes_per_gpu'] / 2**30:.2f} GiB "
            f'cache/GPU, above the packing limit'
        )

# Keep each document's 31 canonical sections together for prefix-cache reuse,
# while evaluating shorter documents first for an early health signal.
section_order = {section: index for index, section in enumerate(CANONICAL_SECTIONS)}
generation_work = sorted(
    work, key=lambda item: (item['source_tokens'], item['document_no'], section_order[item['section']])
)
submission_chunks = [
    generation_work[start:start + SUBMISSION_CHUNK_SIZE]
    for start in range(0, len(generation_work), SUBMISSION_CHUNK_SIZE)
]
run.config.update({
    'test_documents': len(test_df), 'section_requests': len(work),
    'sections_per_document': len(CANONICAL_SECTIONS), 'max_model_length': MAX_MODEL_LENGTH,
    'prompt_tokens_p50': int(pd.Series([x['prompt_tokens_estimate'] for x in work]).median()),
    'prompt_tokens_p95': int(pd.Series([x['prompt_tokens_estimate'] for x in work]).quantile(.95)),
    'prompt_tokens_max': max(x['prompt_tokens_estimate'] for x in work),
    'sequence_token_budget_max': max(x['sequence_token_budget'] for x in work),
    'rag_index_fingerprint': index_fingerprint,
    'rag_input_artifact': prepared_input_artifact_name,
    'rag_train_rows': len(train_df),
    'rag_index_nodes': rag_index_node_count,
})

engine = LLM(
    model=str(MERGED_MODEL_DIR), tokenizer=str(MERGED_MODEL_DIR), dtype='bfloat16',
    hf_overrides={'architectures': ['Qwen3_5ForConditionalGeneration']},
    language_model_only=True, tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LENGTH, gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    kv_cache_memory_bytes=int(cache_budget_per_gpu), max_num_seqs=REQUESTED_BATCH_SIZE,
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS, async_scheduling=True,
    enable_chunked_prefill=True, enable_prefix_caching=True, trust_remote_code=False,
)

def sampling_for(item: dict[str, Any]) -> SamplingParams:
    return SamplingParams(
        max_tokens=item['max_new_tokens'], temperature=TEMPERATURE,
        top_p=TOP_P, top_k=TOP_K, min_p=MIN_P,
        presence_penalty=PRESENCE_PENALTY, repetition_penalty=REPETITION_PENALTY,
        seed=SEED,
    )

print(
    f'vLLM ready: max_model_len={MAX_MODEL_LENGTH:,}, section requests={len(work):,}, '
    f'max live sequences={REQUESTED_BATCH_SIZE}, tensor_parallel={TENSOR_PARALLEL_SIZE}, '
    f'cache budget/GPU={cache_budget_per_gpu / 2**30:.2f} GiB'
)


## 8. Generate grounded JSON for the real RAG test split

Each output row corresponds to one canonical section of one `rag/test` judgment: 68 × 31 = 2,108 requests for the complete split. The local Parquet is rewritten after every document-sized submission. Validation reports whether the completion is JSON, whether it contains exactly the requested section, and what fraction of predicted spans occur verbatim in the test source.


In [ ]:
OUTPUT_COLUMNS = [
    'no', 'dataset_id', 'document_no', 'parent_id', 'section', 'question',
    'corpus', 'source_file', 'source_sha256', 'split',
    'thinking_trace', 'answer', 'raw_output', 'gold_answer',
    'json_valid', 'canonical_schema_valid', 'predicted_section_count',
    'predicted_span_count', 'grounded_span_count', 'grounded_span_rate',
    'retrieved_reference_count', 'retrieved_train_ids_json',
    'retrieved_node_ids_json', 'retrieval_scores_json', 'rag_index_fingerprint',
    'finish_reason', 'stop_reason', 'prompt_tokens', 'prompt_tokens_estimate',
    'source_tokens', 'gold_tokens', 'completion_tokens', 'sequence_token_budget',
    'elapsed_seconds', 'max_new_tokens', 'temperature', 'enable_thinking',
    'max_model_length', 'max_live_sequences', 'submission_chunk_size',
    'estimated_sequence_cache_gib', 'kv_cache_budget_gib', 'gpu_memory_utilization',
]

def split_qwen_output(raw_output: str) -> tuple[str | None, str]:
    if '</think>' not in raw_output:
        return (raw_output.strip(), '') if ENABLE_THINKING else (None, raw_output.strip())
    thinking_trace, answer = raw_output.split('</think>', 1)
    return thinking_trace.removeprefix('<think>').strip(), answer.strip()

def parse_json_answer(answer: str) -> dict[str, Any] | None:
    candidate = answer.strip()
    if candidate.startswith('```'):
        candidate = re.sub(r'^```(?:json)?\s*', '', candidate, flags=re.IGNORECASE)
        candidate = re.sub(r'\s*```$', '', candidate)
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        start, end = candidate.find('{'), candidate.rfind('}')
        if start < 0 or end <= start:
            return None
        try:
            parsed = json.loads(candidate[start:end + 1])
        except json.JSONDecodeError:
            return None
    return parsed if isinstance(parsed, dict) else None

def validate_prediction(answer: str, source_text: str, requested_section: str) -> dict[str, Any]:
    parsed = parse_json_answer(answer)
    if parsed is None:
        return {
            'json_valid': False, 'canonical_schema_valid': False,
            'predicted_section_count': 0, 'predicted_span_count': 0,
            'grounded_span_count': 0, 'grounded_span_rate': 0.0,
        }
    sections = parsed.get('sections') if isinstance(parsed.get('sections'), dict) else {}
    section_value = sections.get(requested_section)
    empty_sections = parsed.get('empty_sections')
    empty_contract_valid = (
        empty_sections == ([requested_section] if section_value == [] else [])
    )
    schema_valid = (
        list(parsed) == ['sections', 'empty_sections']
        and list(sections) == [requested_section]
        and isinstance(section_value, list)
        and empty_contract_valid
    )
    spans = [
        span for span in (section_value if isinstance(section_value, list) else [])
        if isinstance(span, str) and span
    ]
    grounded = sum(span in source_text for span in spans)
    return {
        'json_valid': True,
        'canonical_schema_valid': schema_valid,
        'predicted_section_count': len(sections),
        'predicted_span_count': len(spans),
        'grounded_span_count': grounded,
        'grounded_span_rate': grounded / len(spans) if spans else 1.0,
    }

results: list[dict[str, Any]] = []
processed_tokens = 0
evaluation_started = time.perf_counter()
pd.DataFrame(columns=OUTPUT_COLUMNS).to_parquet(OUTPUT_PARQUET, index=False)
progress = tqdm(total=len(work), desc='RAG test evaluation', unit='document', dynamic_ncols=True)

for chunk_index, chunk in enumerate(submission_chunks):
    chunk_started = time.perf_counter()
    print(
        f'Chunk {chunk_index + 1}/{len(submission_chunks)}: {len(chunk)} documents; '
        f'sequence budgets {min(item["sequence_token_budget"] for item in chunk):,}-'
        f'{max(item["sequence_token_budget"] for item in chunk):,} tokens'
    )
    generated = engine.generate(
        [item['prompt'] for item in chunk],
        [sampling_for(item) for item in chunk],
        use_tqdm=True,
    )
    chunk_elapsed = time.perf_counter() - chunk_started
    if len(generated) != len(chunk):
        raise RuntimeError(f'vLLM returned {len(generated)} outputs for {len(chunk)} prompts')
    for item, request_output in zip(chunk, generated, strict=True):
        if not request_output.outputs:
            raise RuntimeError(f"No completion for {item['row']['id']}")
        completion = request_output.outputs[0]
        raw_output = completion.text
        thinking_trace, answer = split_qwen_output(raw_output)
        row = item['row']
        validation = validate_prediction(answer, row['input_text'], item['section'])
        references = item['references']
        results.append({
            'no': item['no'], 'dataset_id': item['dataset_id'],
            'document_no': item['document_no'], 'parent_id': row['id'],
            'section': item['section'], 'question': item['question'], 'corpus': row['corpus'],
            'source_file': row['source_file'], 'source_sha256': row['source_sha256'],
            'split': row['split'], 'thinking_trace': thinking_trace,
            'answer': answer, 'raw_output': raw_output, 'gold_answer': item['gold_answer'],
            **validation,
            'retrieved_reference_count': len(references),
            'retrieved_train_ids_json': json.dumps(sorted({ref['dataset_id'] for ref in references})),
            'retrieved_node_ids_json': json.dumps([ref['node_id'] for ref in references]),
            'retrieval_scores_json': json.dumps([
                {
                    'section': ref['section'], 'node_id': ref['node_id'],
                    'semantic': ref['semantic_score'], 'structural': ref['structural_score'],
                    'fusion': ref['fusion_score'],
                }
                for ref in references
            ]),
            'rag_index_fingerprint': index_fingerprint,
            'finish_reason': completion.finish_reason,
            'stop_reason': str(completion.stop_reason) if completion.stop_reason is not None else None,
            'prompt_tokens': len(request_output.prompt_token_ids),
            'prompt_tokens_estimate': item['prompt_tokens_estimate'],
            'source_tokens': item['source_tokens'], 'gold_tokens': item['gold_tokens'],
            'completion_tokens': len(completion.token_ids),
            'sequence_token_budget': item['sequence_token_budget'],
            'elapsed_seconds': chunk_elapsed, 'max_new_tokens': item['max_new_tokens'],
            'temperature': TEMPERATURE, 'enable_thinking': ENABLE_THINKING,
            'max_model_length': MAX_MODEL_LENGTH, 'max_live_sequences': REQUESTED_BATCH_SIZE,
            'submission_chunk_size': len(chunk),
            'estimated_sequence_cache_gib': item['estimated_cache_bytes_per_gpu'] / 2**30,
            'kv_cache_budget_gib': cache_budget_per_gpu / 2**30,
            'gpu_memory_utilization': GPU_MEMORY_UTILIZATION,
        })
        processed_tokens += len(request_output.prompt_token_ids) + len(completion.token_ids)
    pd.DataFrame(results, columns=OUTPUT_COLUMNS).sort_values('no').to_parquet(OUTPUT_PARQUET, index=False)
    elapsed = time.perf_counter() - evaluation_started
    rate = processed_tokens / elapsed if elapsed else 0.0
    remaining_tokens = sum(
        future['prompt_tokens_estimate'] + future['max_new_tokens']
        for future_chunk in submission_chunks[chunk_index + 1:]
        for future in future_chunk
    )
    eta = remaining_tokens / (rate * THROUGHPUT_SAFETY_FACTOR) if rate else 0.0
    progress.update(len(chunk))
    progress.set_postfix_str(
        f'elapsed={tqdm.format_interval(elapsed)}, ETA={tqdm.format_interval(eta)}, '
        f'grounded={pd.DataFrame(results).grounded_span_rate.mean():.1%}'
    )
    print(f'Backed up {len(results):,}/{len(work):,} section requests -> {OUTPUT_PARQUET}')
progress.close()

evaluation_elapsed_seconds = time.perf_counter() - evaluation_started
final_results = pd.DataFrame(results, columns=OUTPUT_COLUMNS).sort_values('no')
if len(final_results) != len(work) or final_results['dataset_id'].duplicated().any():
    raise RuntimeError('Final RAG output is incomplete or contains duplicate test IDs')
expected_ids = {f'{parent_id}::{section}' for parent_id in test_df['id'] for section in CANONICAL_SECTIONS}
if set(final_results['dataset_id']) != expected_ids:
    raise RuntimeError('Final output IDs differ from the selected rag/test section IDs')
if not final_results.groupby('parent_id')['section'].apply(list).map(lambda value: value == CANONICAL_SECTIONS).all():
    raise RuntimeError('At least one test document lacks the 31 canonical sections in order')
final_results.to_parquet(OUTPUT_PARQUET, index=False)
run.summary.update({
    'evaluation_elapsed_seconds': evaluation_elapsed_seconds,
    'evaluation_rows_completed': len(final_results),
    'evaluation_documents_completed': int(final_results['parent_id'].nunique()),
    'sections_per_document': len(CANONICAL_SECTIONS),
    'json_valid_rate': float(final_results['json_valid'].mean()),
    'canonical_schema_valid_rate': float(final_results['canonical_schema_valid'].mean()),
    'mean_grounded_span_rate': float(final_results['grounded_span_rate'].mean()),
})
print(
    f'Finished {len(final_results):,} section requests for {final_results.parent_id.nunique()} rag/test documents '
    f'in {evaluation_elapsed_seconds / 3600:.2f} hours; '
    f'JSON valid={final_results.json_valid.mean():.1%}; '
    f'schema valid={final_results.canonical_schema_valid.mean():.1%}; '
    f'mean grounded spans={final_results.grounded_span_rate.mean():.1%}'
)


## 9. Upload the exact RAG test-result Parquet to W&B

This final cell verifies the 2,108-row section-level schema, uploads the same local backup file, records the train-index artifact and split invariants, and then offers the file for download.


In [ ]:
parquet_data = pd.read_parquet(OUTPUT_PARQUET)
if parquet_data.columns.tolist() != OUTPUT_COLUMNS:
    raise RuntimeError(f'Parquet columns differ from OUTPUT_COLUMNS: {parquet_data.columns.tolist()}')
if set(parquet_data['split']) != {EVALUATION_SPLIT}:
    raise RuntimeError('Output contains rows outside rag/test')
expected_output_rows = len(test_df) * len(CANONICAL_SECTIONS)
if len(parquet_data) != expected_output_rows:
    raise RuntimeError(f'Expected {expected_output_rows:,} section rows, found {len(parquet_data):,}')
if set(parquet_data['parent_id']) & set(train_df['id']):
    raise RuntimeError('Output parent IDs overlap the rag/train reference corpus')
if parquet_data.groupby('parent_id')['section'].nunique().ne(len(CANONICAL_SECTIONS)).any():
    raise RuntimeError('At least one rag/test document does not have 31 section outputs')

parquet_artifact = wandb.Artifact(
    name='qwen3-5-4b-rag-test-section-results-parquet',
    type='dataset',
    description='One grounded Qwen3.5 result per canonical section of every rag/test document, augmented only with the matching reference pre-retrieved from rag/train.',
    metadata={
        'row_count': len(parquet_data), 'columns': OUTPUT_COLUMNS,
        'dataset_repo': DATASET_REPO, 'dataset_config': DATASET_CONFIG,
        'reference_split': REFERENCE_SPLIT, 'evaluation_split': EVALUATION_SPLIT,
        'reference_row_count': len(train_df), 'evaluation_document_count': len(test_df),
        'sections_per_document': len(CANONICAL_SECTIONS), 'evaluation_request_count': len(parquet_data),
        'rag_index_fingerprint': index_fingerprint,
        'rag_input_artifact': prepared_input_artifact_name,
        'rag_index_node_count': rag_index_node_count,
        'retrieval_method': prepared_summary['retrieval_method'],
        'test_prompt_source': 'full input_text per section; target_json used only for offline gold scoring/budget, never prompt content',
        'evaluation_elapsed_seconds': evaluation_elapsed_seconds,
        'json_valid_rate': float(parquet_data['json_valid'].mean()),
        'canonical_schema_valid_rate': float(parquet_data['canonical_schema_valid'].mean()),
        'mean_grounded_span_rate': float(parquet_data['grounded_span_rate'].mean()),
    },
)
parquet_artifact.add_file(str(OUTPUT_PARQUET), name=OUTPUT_PARQUET.name)
logged_artifact = run.log_artifact(parquet_artifact, aliases=['latest'])
logged_artifact.wait()
print(f'W&B result artifact: {logged_artifact.name}')
run.finish()

saved = pd.read_parquet(OUTPUT_PARQUET)
display(saved[['no', 'corpus', 'source_file', 'json_valid', 'canonical_schema_valid', 'grounded_span_rate']].head())
if IN_COLAB:
    from google.colab import files
    files.download(str(OUTPUT_PARQUET))
else:
    print(f'Copy this file before deleting the runtime: {OUTPUT_PARQUET.resolve()}')


In [ ]:
# 10. Post-run RAG integrity and held-out test diagnostics
rag_audit_source = pd.read_parquet(OUTPUT_PARQUET).copy()
expected_test_ids = set(test_df['id'])
reference_train_ids = set(train_df['id'])
if set(rag_audit_source['dataset_id']) != expected_test_ids:
    raise RuntimeError('RAG outputs are not exactly the selected rag/test rows')
if set(rag_audit_source['split']) != {EVALUATION_SPLIT}:
    raise RuntimeError('RAG outputs contain a split other than rag/test')
if rag_audit_source['rag_index_fingerprint'].nunique() != 1 or rag_audit_source['rag_index_fingerprint'].iloc[0] != index_fingerprint:
    raise RuntimeError('RAG output fingerprint does not match the rag/train vector index')

retrieved_id_sets = rag_audit_source['retrieved_train_ids_json'].map(lambda value: set(json.loads(value)))
invalid_reference_ids = set().union(*retrieved_id_sets) - reference_train_ids
if invalid_reference_ids:
    raise RuntimeError(f'Retrieval provenance contains non-train IDs: {sorted(invalid_reference_ids)[:5]}')
if (rag_audit_source['retrieved_reference_count'] <= 0).any():
    raise RuntimeError('At least one rag/test output was generated without retrieved train context')

def labelled_spans(value: str) -> set[tuple[str, str]]:
    parsed = parse_json_answer(value)
    if parsed is None:
        return set()
    sections = parsed.get('sections') if isinstance(parsed.get('sections'), dict) else parsed
    return {
        (section, span)
        for section in CANONICAL_SECTIONS
        for span in sections.get(section, [])
        if isinstance(span, str) and span
    }

diagnostic_rows = []
for row in rag_audit_source.itertuples(index=False):
    predicted = labelled_spans(row.answer)
    expected = labelled_spans(row.gold_answer)
    true_positive = len(predicted & expected)
    false_positive = len(predicted - expected)
    false_negative = len(expected - predicted)
    diagnostic_rows.append({
        'dataset_id': row.dataset_id,
        'true_positive_spans': true_positive,
        'false_positive_spans': false_positive,
        'false_negative_spans': false_negative,
        'retrieved_reference_count': row.retrieved_reference_count,
        'grounded_span_rate': row.grounded_span_rate,
    })
rag_test_diagnostics = pd.DataFrame(diagnostic_rows)
tp = int(rag_test_diagnostics['true_positive_spans'].sum())
fp = int(rag_test_diagnostics['false_positive_spans'].sum())
fn = int(rag_test_diagnostics['false_negative_spans'].sum())
micro_precision = tp / (tp + fp) if tp + fp else 1.0
micro_recall = tp / (tp + fn) if tp + fn else 1.0
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if micro_precision + micro_recall else 0.0
print(
    f'RAG integrity passed: {len(rag_audit_source)} rag/test rows; all provenance is from rag/train. '
    f'Exact labelled-span micro P/R/F1 = {micro_precision:.3f}/{micro_recall:.3f}/{micro_f1:.3f}'
)
display(rag_test_diagnostics.head())
